# GRPO Tuning: Qwen3.5-4B — unsloth + LoRA

## 1. Install (Qwen3.5 + unsloth + trl 0.24) — 실행 후 런타임 재시작

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        --refresh-package unsloth_zoo --reinstall-package unsloth_zoo \
        --refresh-package unsloth    --reinstall-package unsloth \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.24.0
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
!apt-get update -qq && apt-get install -y -qq libz3-dev

## 2. Mount Drive & HuggingFace login

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
from huggingface_hub import login

HF_TOKEN    = userdata.get('HF_TOKEN')
HF_REPO_IN  = 'minsu0567/IAD-X1-SFT-answer-last'
HF_REPO_OUT = 'minsu0567/IAD-X1-GRPO-answer-last-no-hard'
login(token=HF_TOKEN)
print('HF login OK.')

## 3. GPU check

In [ ]:
!nvidia-smi

## 4. Paths & sys.path

In [ ]:
import os, sys

DRIVE_ROOT = '/content/drive/MyDrive'
IAD_R1_DIR = f'{DRIVE_ROOT}/IAD-R1-main'
GRPO_SRC   = f'{DRIVE_ROOT}/IAD-X1/grpo_src'
DATA_JSON  = f'{DRIVE_ROOT}/grpo_no_hard_samples_answer_last.json'
IMG_DIRS   = [f'{DRIVE_ROOT}/PA-SFT_dataset_2', f'{DRIVE_ROOT}/GRPO_dataset3']
OUTPUT_DIR = '/content/GRPO_output/Qwen3_5_4B'

for p in [IAD_R1_DIR, GRPO_SRC] + IMG_DIRS:
    assert os.path.isdir(p), f'Missing dir: {p}'
assert os.path.isfile(DATA_JSON), f'Missing: {DATA_JSON}'
assert os.path.isfile(f'{GRPO_SRC}/reward_c_qwen_answer_last.py')
assert os.path.isfile(f'{GRPO_SRC}/grpo_pipeline.py')

if GRPO_SRC not in sys.path:
    sys.path.insert(0, GRPO_SRC)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Paths OK. Output:', OUTPUT_DIR)

## 5. Config

In [ ]:
import os

TYPE_REWARD_MODE = 'graded'
os.environ['TYPE_REWARD_MODE'] = TYPE_REWARD_MODE

LEARNING_RATE      = 1e-5
NUM_GENERATIONS    = 4
MAX_STEPS          = 489
SAVE_STEPS         = MAX_STEPS
MAX_PROMPT_LEN     = 8192
MAX_COMPLETION_LEN = 640
IMG_RESOLUTION     = 512
MAX_SAMPLES        = None
BETA               = 0.01
MAX_SEQ_LENGTH     = 16384

LORA_R       = 64
LORA_ALPHA   = 64
LORA_DROPOUT = 0
RANDOM_STATE = 3407

print(f'num_gen={NUM_GENERATIONS}, max_steps={MAX_STEPS}, TYPE_REWARD_MODE={TYPE_REWARD_MODE}')

## 6. Load SFT model + add LoRA

In [ ]:
import shutil
shutil.rmtree('/content/unsloth_compiled_cache', ignore_errors=True)

from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    model_name     = HF_REPO_IN,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = False,
    fast_inference = False,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = LORA_R, lora_alpha = LORA_ALPHA, lora_dropout = LORA_DROPOUT, bias = "none",
    random_state = RANDOM_STATE, use_rslora = False, loftq_config = None,
    use_gradient_checkpointing = "unsloth",
)
print('Model + LoRA ready.')

## 7. Dataset

In [ ]:
from grpo_pipeline import strip_think_primer, build_dataset

strip_think_primer(tokenizer)
train_dataset = build_dataset(DATA_JSON, img_resolution=IMG_RESOLUTION, max_samples=MAX_SAMPLES)
print(train_dataset)

## 8. Trainer

In [ ]:
from grpo_pipeline import build_trainer

trainer = build_trainer(
    model, tokenizer, train_dataset, OUTPUT_DIR,
    learning_rate     = LEARNING_RATE,
    num_generations   = NUM_GENERATIONS,
    max_steps         = MAX_STEPS,
    save_steps        = SAVE_STEPS,
    max_prompt_len    = MAX_PROMPT_LEN,
    max_completion_len= MAX_COMPLETION_LEN,
    beta              = BETA,
)
print('Trainer ready.')

## 9. Train

In [ ]:
trainer.train()

## 10. Save LoRA & push to Hub

In [ ]:
model.push_to_hub_merged(HF_REPO_OUT, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
print('Pushed merged 16bit ->', HF_REPO_OUT)

In [ ]:
ADAPTER_REPO = 'minsu0567/IAD-X1-answer_last-no-hard-adapter'
model.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)
print('Pushed adapter ->', ADAPTER_REPO)